# 1. Data Loading and Feature Engineering

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

In [ ]:
df = pd.read_csv('dataset_with_regime.csv', parse_dates=['observation_date'])
print(f"Dataset: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

In [ ]:
df['lag1'] = df['oil_return'].shift(1)
df['lag2'] = df['oil_return'].shift(2)
df['lag_price'] = df['WTI_Price'].shift(1)

# Target: log return (predict next-day return, then convert to price)
df['target_return'] = np.log(df['WTI_Price_next'] / df['WTI_Price'])

# Drop rows with NaN from lag creation
df = df.dropna().copy()

print(f"After lag features: {df.shape[0]} rows")

In [ ]:
split_idx = int(len(df) * 0.8)

train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

print(f"Train: {len(train_df)} rows ({train_df['observation_date'].min().date()} to {train_df['observation_date'].max().date()})")
print(f"Test:  {len(test_df)} rows ({test_df['observation_date'].min().date()} to {test_df['observation_date'].max().date()})")

# 2. Baseline XGBoost Model

In [ ]:
baseline_features = [
    'Brent_Price', 'DTWEXBGS', 'EFFR', 'Gas_Price', 'Gold_Price', 'OVX_Close',
    'WCESTUS1', 'supply_shock', 'supply_shock_pct',
    'GPR', 'GPRT', 'GPRC', 'GEPUCURRENT', 'CPILFESL',
    'INDPRO', 'industrial_production_growth',
    'lag1', 'lag2', 'lag_price'
]

enhanced_features = baseline_features + ['rolling_entropy', 'hidden_state', 'regime_label']

target_col = 'target_return'

In [ ]:
xgb_params = dict(
    n_estimators=5000,
    learning_rate=0.02,
    max_depth=3,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_alpha=0.5,
    reg_lambda=2,
    random_state=42,
    objective='reg:squarederror'
)

model_baseline = xgb.XGBRegressor(**xgb_params)
model_baseline.fit(train_df[baseline_features], train_df[target_col])

y_pred_baseline_train = model_baseline.predict(train_df[baseline_features])
y_pred_baseline_test = model_baseline.predict(test_df[baseline_features])

price_pred_baseline = test_df['WTI_Price'].values * np.exp(y_pred_baseline_test)
price_true = test_df['WTI_Price_next'].values

rmse_baseline = np.sqrt(mean_squared_error(price_true, price_pred_baseline))
mae_baseline = mean_absolute_error(price_true, price_pred_baseline)
mape_baseline = np.mean(np.abs((price_true - price_pred_baseline) / price_true)) * 100

print("=== Baseline XGBoost ===")
print(f"RMSE: {rmse_baseline:.4f}")
print(f"MAE:  {mae_baseline:.4f}")
print(f"MAPE: {mape_baseline:.4f}%")

# 3. Enhanced XGBoost Model (with Regime Features)

In [ ]:
model_enhanced = xgb.XGBRegressor(**xgb_params)
model_enhanced.fit(train_df[enhanced_features], train_df[target_col])

y_pred_enhanced_test = model_enhanced.predict(test_df[enhanced_features])

price_pred_enhanced = test_df['WTI_Price'].values * np.exp(y_pred_enhanced_test)

rmse_enhanced = np.sqrt(mean_squared_error(price_true, price_pred_enhanced))
mae_enhanced = mean_absolute_error(price_true, price_pred_enhanced)
mape_enhanced = np.mean(np.abs((price_true - price_pred_enhanced) / price_true)) * 100

print("=== Enhanced XGBoost (with Regime Features) ===")
print(f"RMSE: {rmse_enhanced:.4f}")
print(f"MAE:  {mae_enhanced:.4f}")
print(f"MAPE: {mape_enhanced:.4f}%")

# 4. Model Comparison

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Baseline XGBoost', 'Enhanced XGBoost (+ Regime)'],
    'RMSE': [rmse_baseline, rmse_enhanced],
    'MAE': [mae_baseline, mae_enhanced],
    'MAPE (%)': [mape_baseline, mape_enhanced]
}).round(4)

comparison.loc[2] = [
    'Improvement (%)',
    f"{(rmse_baseline - rmse_enhanced) / rmse_baseline * 100:.2f}%",
    f"{(mae_baseline - mae_enhanced) / mae_baseline * 100:.2f}%",
    f"{(mape_baseline - mape_enhanced) / mape_baseline * 100:.2f}%"
]

comparison

In [ ]:
test_dates = test_df['observation_date'].values

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(test_dates, price_true, color='black', linewidth=1.5, label='Actual', alpha=0.9)
ax.plot(test_dates, price_pred_baseline, color='steelblue', linewidth=1,
        label='Baseline XGBoost', alpha=0.7, linestyle='--')
ax.plot(test_dates, price_pred_enhanced, color='darkorange', linewidth=1,
        label='Enhanced XGBoost', alpha=0.7, linestyle='--')
ax.set_xlabel('Date')
ax.set_ylabel('WTI Price (USD/barrel)')
ax.set_title('WTI Price Prediction: Baseline vs Regime-Enhanced XGBoost')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

imp_baseline = pd.Series(model_baseline.feature_importances_,
                          index=baseline_features).sort_values()
imp_baseline.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Baseline XGBoost')
axes[0].set_xlabel('Feature Importance')

imp_enhanced = pd.Series(model_enhanced.feature_importances_,
                          index=enhanced_features).sort_values()
colors = ['darkorange' if f in ['rolling_entropy', 'hidden_state', 'regime_label']
          else 'steelblue' for f in imp_enhanced.index]
imp_enhanced.plot(kind='barh', ax=axes[1], color=colors)
axes[1].set_title('Enhanced XGBoost (regime features in orange)')
axes[1].set_xlabel('Feature Importance')

fig.suptitle('Feature Importance Comparison', fontsize=14)
plt.tight_layout()
plt.show()

## Summary

**Part II Findings:**
- The Enhanced XGBoost model incorporates three regime-derived features: `rolling_entropy`, `hidden_state`, and `regime_label`.
- Comparison metrics (RMSE, MAE, MAPE) are reported in the table above.
- The feature importance plot shows how regime features rank relative to traditional market variables.